# 009: Dropout Regularization — Bernoulli masking, ensemble interpretation, and inverted dropout

## 1. THE OVERFITTING PROBLEM
- **Overfitting**: A model with too many parameters memorizes training noise instead of learning generalizable patterns.
- **Dropout (Srivastava et al., 2014)**: During training, randomly **zero out** each neuron's activation with probability $p$ (the drop rate).

## 2. MATHEMATICAL INTERPRETATION
- **Bernoulli Mask**: For each neuron $j$ in layer $l$, sample a mask $d_j^{[l]} \sim \text{Bernoulli}(1 - p)$:
  $$\tilde{a}_j^{[l]} = d_j^{[l]} \cdot a_j^{[l]}$$
- **Ensemble Interpretation**: Dropout is equivalent to training an exponential number of sub-networks (each with different neurons removed) and averaging their predictions at test time.

## 3. INVERTED DROPOUT
- **Problem**: During test time, all neurons are active. The expected activation magnitude is larger by a factor of $\frac{1}{1-p}$ compared to training.
- **Inverted Dropout Solution**: During **training**, scale surviving activations by $\frac{1}{1-p}$:
  $$\tilde{a}_j^{[l]} = \frac{d_j^{[l]} \cdot a_j^{[l]}}{1 - p}$$
- This ensures test-time expectations match training expectations, requiring **no modification at inference**.

---


In [ ]:
import numpy as np

class DropoutLayer:
    """From-scratch inverted dropout implementation."""
    def __init__(self, drop_rate=0.5):
        self.p = drop_rate  # Probability of dropping a neuron
        self.mask = None

    def forward(self, A, training=True):
        """Apply inverted dropout during training, pass through during inference."""
        if training:
            # Generate Bernoulli mask: 1 with prob (1-p), 0 with prob p
            self.mask = (np.random.rand(*A.shape) > self.p).astype(float)
            # Apply mask and scale
            return (A * self.mask) / (1.0 - self.p)
        else:
            # No modification at inference (inverted dropout handles scaling during training)
            return A



In [ ]:
print("--- Inverted Dropout Demo ---")
np.random.seed(42)
A = np.ones((5, 8))  # 5 neurons, 8 samples (all activations = 1.0)

dropout = DropoutLayer(drop_rate=0.5)

# Training mode: ~50% of neurons zeroed, survivors scaled by 2x
A_train = dropout.forward(A, training=True)
print(f"Training output (row 0): {A_train[0].round(2)}")
print(f"Training mean per neuron: {A_train.mean(axis=1).round(2)}")

# Inference mode: all neurons active, no scaling needed
A_test = dropout.forward(A, training=False)
print(f"\nInference output (row 0): {A_test[0].round(2)}")
print(f"Inference mean per neuron: {A_test.mean(axis=1).round(2)}")
print("\n[Key Insight] Training mean ≈ 1.0 ≈ Inference mean (inverted dropout preserves expectation).")



In [ ]:
# ── Statistical verification over many trials ──
print("\n--- Statistical Expectation Check (1000 trials) ---")
means = []
for _ in range(1000):
    out = dropout.forward(A, training=True)
    means.append(out.mean())
print(f"Average activation across 1000 dropout trials: {np.mean(means):.4f}")
print(f"Expected: 1.0000 (inverted dropout ensures E[output] = E[input])")
